# PMD — Examen Práctico (Atletismo) · Solo PySpark (sin SQL)

**Duración orientativa:** 50–60 minutos  
**Reglas:** Solo se permiten las APIs de `pyspark.sql` (DataFrames) y **RDDs** **cuando se indique**.  
**Datasets sintéticos** del dominio de **atletismo**:

- `athletes(athlete_id, name, country)`
- `events(event_id, event_name, category)`  → ej.: `100m`, `Long Jump`, categorías: `Track`, `Field`
- `results(event_id, athlete_id, date, time_sec, distance_m, points)`  → según evento se usa `time_sec` o `distance_m`; `points` es unificado

> En cada ejercicio solo puedes usar **dos DataFrames** (indicados en cada enunciado).


> **Versión ALUMNO** — Enunciados con celdas `TODO` para completar.

## 0) Setup
Ejecuta esta celda antes de empezar.

In [23]:
# !apt-get install openjdk-11-jdk -y
# !pip install pyspark

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql import Window

spark = SparkSession.builder.appName("PMD-Examen-Atletismo").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

# ---- Datasets (sintéticos) ----
athletes_schema = T.StructType([
    T.StructField("athlete_id", T.StringType(), False),
    T.StructField("name",       T.StringType(), True),
    T.StructField("country",    T.StringType(), True),
])
athletes_data = [
    ("A001","Alex Ruiz","ESP"),
    ("A002","Bea López","ESP"),
    ("A003","Chris Nolan","GBR"),
    ("A004","Dina Akua","GHA"),
]

events_schema = T.StructType([
    T.StructField("event_id",   T.StringType(), False),
    T.StructField("event_name", T.StringType(), True),
    T.StructField("category",   T.StringType(), True),  # Track / Field
])
events_data = [
    ("E100","100m","Track"),
    ("E200","200m","Track"),
    ("E-LJ","Long Jump","Field"),
    ("E-SP","Shot Put","Field"),
]

results_schema = T.StructType([
    T.StructField("event_id",   T.StringType(), False),
    T.StructField("athlete_id", T.StringType(), False),
    T.StructField("date_str",   T.StringType(), False),
    T.StructField("time_sec",   T.DoubleType(), True),
    T.StructField("distance_m", T.DoubleType(), True),
    T.StructField("points",     T.IntegerType(), True),
])
results_data = [
    ("E100","A001","2025-10-01",10.90,None, 900),
    ("E100","A002","2025-10-01",11.20,None, 850),
    ("E100","A003","2025-10-01",10.75,None, 930),
    ("E200","A001","2025-10-01",22.10,None, 880),
    ("E-LJ","A002","2025-10-01",None,6.10,  840),
    ("E-LJ","A003","2025-10-01",None,7.05,  950),
    ("E-SP","A004","2025-10-01",None,16.3,  910),
    ("E100","A001","2025-10-02",10.85,None, 910),
    ("E200","A003","2025-10-02",21.80,None, 900),
    ("E-LJ","A004","2025-10-02",None,6.90,  880),
]

athletes = spark.createDataFrame(athletes_data, athletes_schema)
events   = spark.createDataFrame(events_data, events_schema)
results  = (spark.createDataFrame(results_data, results_schema)
            .withColumn("date", F.to_date("date_str"))
            .drop("date_str"))

print("athletes:"); athletes.show(truncate=False)
print("events:"); events.show(truncate=False)
print("results:"); results.show(truncate=False)


athletes:
+----------+-----------+-------+
|athlete_id|name       |country|
+----------+-----------+-------+
|A001      |Alex Ruiz  |ESP    |
|A002      |Bea López  |ESP    |
|A003      |Chris Nolan|GBR    |
|A004      |Dina Akua  |GHA    |
+----------+-----------+-------+

events:
+--------+----------+--------+
|event_id|event_name|category|
+--------+----------+--------+
|E100    |100m      |Track   |
|E200    |200m      |Track   |
|E-LJ    |Long Jump |Field   |
|E-SP    |Shot Put  |Field   |
+--------+----------+--------+

results:
+--------+----------+--------+----------+------+----------+
|event_id|athlete_id|time_sec|distance_m|points|date      |
+--------+----------+--------+----------+------+----------+
|E100    |A001      |10.9    |NULL      |900   |2025-10-01|
|E100    |A002      |11.2    |NULL      |850   |2025-10-01|
|E100    |A003      |10.75   |NULL      |930   |2025-10-01|
|E200    |A001      |22.1    |NULL      |880   |2025-10-01|
|E-LJ    |A002      |NULL    |6.1      

---
## 1) Básico — 2025-10-01: **puntos totales por atleta** (Top-3)
**Usa SOLO** `results` + `athletes`.

1.a) **DataFrames**: Filtra por `date == 2025-10-01`, une con `athletes` por `athlete_id`, agrega **puntos totales por atleta**, ordena desc y muestra **Top-3**.  
1.b) **RDDs**: Mismo objetivo **con RDDs** (Pair RDD por `athlete_id`).


### 1.a) DataFrames — `TODO`

> Pistas DF: `filter` → `join` → `groupBy("athlete_id","name")` → `sum("points")` → `orderBy(desc)` → `limit(3)`  

In [17]:
# RESULT

+----------+-----------+------------+
|athlete_id|name       |total_points|
+----------+-----------+------------+
|A003      |Chris Nolan|1880        |
|A001      |Alex Ruiz  |1780        |
|A002      |Bea López  |1690        |
+----------+-----------+------------+



In [ ]:
# TODO: DataFrames
from pyspark.sql import functions as F

# YOUR CODE HERE

### 1.b) RDDs — `TODO`

> Pistas RDD: `filter + (athlete_id, points)` + `(athlete_id, name)` → `leftOuterJoin` → `reduceByKey` → `takeOrdered(3)`



In [20]:
# RESULT

Top-3 atletas por puntos (RDD): [('Chris Nolan', 1880), ('Alex Ruiz', 1780), ('Bea López', 1690)]
El atleta Chris Nolan tiene un total de 1880 puntos.
El atleta Alex Ruiz tiene un total de 1780 puntos.
El atleta Bea López tiene un total de 1690 puntos.


In [ ]:
# TODO: RDDs (Pair RDD join)
from datetime import date

# YOUR CODE HERE

---
## 2) Intermedio — 2025-10-01: **puntos medios y totales por categoría de evento**
**Usa SOLO** `results` + `events` (sin SQL).

- Filtra por día
- Une por `event_id`.
- Calcula **media** (F.avg -> avg_points) y **suma** (F.sum -> total_points) de `points` por `category`.
- Ordena por `total_points` desc.
- Muestra todas las categorías presentes.


In [21]:
# RESULT

+--------+----------+------------+
|category|avg_points|total_points|
+--------+----------+------------+
|Track   |890.0     |3560        |
|Field   |900.0     |2700        |
+--------+----------+------------+



In [ ]:
# TODO: DataFrames
from pyspark.sql import functions as F

# YOUR CODE HERE

---
## 3) Avanzado — 2025-10-01: **Top-2 eventos por atleta** según puntos
**Usa SOLO** `results` + `events` y **ventanas** (sin SQL).

- Filtra por día (F.col)
- Une por `event_id`.
- Agrega puntos por `(athlete_id, event_name)`.
- Aplica ventana `partitionBy('athlete_id').orderBy(desc('total_points'), asc('event_name'))`. Usa F.desc y F.asc según corresponda.
- Calcula ranking con F.row_number() sobre la ventana
- Mantén `raking <= 2` y ordena por `athlete_id, ranking`.


In [22]:
# RESULT

+----------+----------+------------+-------+
|athlete_id|event_name|total_points|ranking|
+----------+----------+------------+-------+
|A001      |100m      |900         |1      |
|A001      |200m      |880         |2      |
|A002      |100m      |850         |1      |
|A002      |Long Jump |840         |2      |
|A003      |Long Jump |950         |1      |
|A003      |100m      |930         |2      |
|A004      |Shot Put  |910         |1      |
+----------+----------+------------+-------+



In [ ]:
# TODO: DataFrames + Window
from pyspark.sql import functions as F, Window

# YOUR CODE HERE